# Forecasting Model Comparison and Complexity

This notebook addresses two separate research questions using the saved trade-only results: **RQ1**, how machine learning compares with classical forecasting methods, and **RQ4**, whether greater model complexity accompanies better held-out performance. March 2026 is the final test month.

No models are fitted. Existing result CSVs and the saved February validation tables in notebooks 11 and 12 supply the evidence. REER and Suez experiments are outside this comparison. Tables and four figures are saved directly under `results/`.

## 1. Load Benchmark Results

Combine the baseline, ETS, ARIMA, XGBoost and LSTM result tables without overwriting them. Retain selected configuration labels while using consistent model-family names in figures. Check the forecast month, metric values, and series counts against the saved March targets.

In [ ]:
from pathlib import Path
import sys
import ast
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_loading import COUNTRY_PREFIXES, DATA_DIR, SERIES_KEYS

RESULTS_DIR = PROJECT_ROOT / "results"
COUNTRIES = list(COUNTRY_PREFIXES)
MODEL_ORDER = ["Naive", "Seasonal Naive", "ETS", "ARIMA", "XGBoost", "LSTM"]
METRICS = ["MAE", "RMSE", "sMAPE (%)"]
MODEL_COLORS = {
    "Naive": "#7f7f7f", "Seasonal Naive": "#bdbdbd",
    "ETS": "#1f77b4", "ARIMA": "#17becf",
    "XGBoost": "#ff7f0e", "LSTM": "#9467bd",
}
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.2, "axes.axisbelow": True,
})

result_names = ["baseline", "statistical", "arima", "xgboost", "lstm"]
benchmark = pd.concat(
    [pd.read_csv(RESULTS_DIR / f"{name}_results.csv") for name in result_names],
    ignore_index=True,
)
benchmark["Model label"] = benchmark["Model"]

def model_family(label):
    if label in MODEL_ORDER:
        return label
    for family in ["ARIMA", "XGBoost", "LSTM"]:
        if label.startswith(family):
            return family
    raise ValueError(f"Unknown model label: {label}")

benchmark["Model"] = benchmark["Model label"].map(model_family)
assert benchmark["Forecast Month"].eq("2026-03").all()
assert set(benchmark.Country) == set(COUNTRIES)
assert not benchmark.duplicated(["Country", "Model"]).any()
assert len(benchmark) == len(COUNTRIES) * len(MODEL_ORDER)
assert np.isfinite(benchmark[METRICS].to_numpy()).all()
assert benchmark[METRICS].ge(0).all().all()
assert benchmark["sMAPE (%)"].le(200 + 1e-8).all()

for country, prefix in COUNTRY_PREFIXES.items():
    group = benchmark.loc[benchmark.Country.eq(country)]
    assert set(group.Model) == set(MODEL_ORDER)
    target = pd.read_parquet(
        DATA_DIR / "forecasting" / f"{prefix}_test.parquet",
        columns=SERIES_KEYS + ["Month"],
    )
    assert target.Month.eq(pd.Timestamp("2026-03-01")).all()
    assert not target.duplicated(SERIES_KEYS + ["Month"]).any()
    assert group["N Series"].eq(len(target)).all()

display(benchmark[
    ["Country", "Model", "Model label", "N Series", "MAE", "RMSE", "sMAPE (%)"]
])

## 2. RQ1 — March Forecast Accuracy

Compare all six selected benchmark methods using sMAPE, the primary reporting metric. All country panels use the same vertical scale and model colours; lower bars indicate lower errors. Labels show observed scores, not uncertainty intervals. Comparisons concern March 2026 on each country's existing cohort, rather than statistical significance or performance in every future month.

In [ ]:
fig, axes = plt.subplots(1, len(COUNTRIES), figsize=(15, 5), sharey=True)
upper = max(float(benchmark["sMAPE (%)"].max()) * 1.18, 1.0)
for ax, country in zip(axes, COUNTRIES):
    values = benchmark.loc[benchmark.Country.eq(country)].set_index(
        "Model"
    ).reindex(MODEL_ORDER)["sMAPE (%)"]
    bars = ax.bar(
        range(len(MODEL_ORDER)), values,
        color=[MODEL_COLORS[model] for model in MODEL_ORDER],
    )
    ax.bar_label(bars, labels=[f"{value:.1f}" for value in values], padding=3, fontsize=9)
    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right")
    ax.set_ylim(0, upper)
    ax.set_title(country)
    ax.set_xlabel("Model")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("March sMAPE (%)")
fig.suptitle("RQ1: trade-only forecasting accuracy — March 2026", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "rq1_smape_comparison.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

## 3. RQ1 — Rankings Across Evaluation Metrics

Rank models separately within each country for MAE, RMSE and sMAPE. Rank 1 is the lowest error; ties receive the same minimum rank. The common colour scale highlights agreement and changes in model ordering across metrics. Rankings conceal the size of accuracy differences, so interpret this figure alongside the numerical table and sMAPE chart. Raw trade-value errors are not averaged across countries.

In [ ]:
rank_columns = []
for metric in METRICS:
    column = f"{metric} rank"
    benchmark[column] = benchmark.groupby("Country")[metric].rank(method="min")
    rank_columns.append(column)

fig, axes = plt.subplots(
    1, len(COUNTRIES), figsize=(13, 5), sharey=True, layout="constrained"
)
for ax, country in zip(axes, COUNTRIES):
    ranks = benchmark.loc[benchmark.Country.eq(country)].set_index(
        "Model"
    ).reindex(MODEL_ORDER)[rank_columns]
    image = ax.imshow(
        ranks.to_numpy(), cmap="RdYlGn_r",
        vmin=1, vmax=len(MODEL_ORDER), aspect="auto",
    )
    for row in range(len(MODEL_ORDER)):
        for column in range(len(METRICS)):
            value = ranks.iloc[row, column]
            colour = "white" if value <= 2 or value >= 5 else "black"
            ax.text(column, row, f"{value:.0f}", ha="center", va="center", color=colour)
    ax.set_xticks(range(len(METRICS)))
    ax.set_xticklabels(["MAE", "RMSE", "sMAPE"])
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels(MODEL_ORDER)
    ax.set_title(country)
    ax.grid(False)
fig.colorbar(image, ax=list(axes), ticks=range(1, 7), label="Rank (1 = lowest error)")
fig.suptitle("RQ1: model rankings across metrics", fontsize=13)
fig.savefig(RESULTS_DIR / "rq1_metric_rankings.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

benchmark = benchmark.sort_values(["Country", "Model"]).reset_index(drop=True)
benchmark.to_csv(RESULTS_DIR / "rq1_benchmark_results.csv", index=False)
display(benchmark[["Country", "Model"] + METRICS + rank_columns])

for country in COUNTRIES:
    group = benchmark.loc[benchmark.Country.eq(country)]
    winners = []
    for metric in METRICS:
        names = group.loc[group[metric].eq(group[metric].min()), "Model"].tolist()
        winners.append(f"{metric}: {', '.join(names)}")
    print(f"{country} — lowest March errors: {'; '.join(winners)}.")

## 4. Load Configuration and Validation Records

Read literal model settings and displayed February validation tables from notebooks 11 and 12 without executing their training cells. Check that the saved March configurations match the February MAE selections and that validation and test counts agree.

Displayed validation scores can be rounded; they are reused at their saved precision. The configuration-level analysis covers XGBoost and LSTM because their candidate validation records are available. ETS and ARIMA remain in the RQ1 benchmark.

In [ ]:
def read_notebook_setting(notebook_name, variable_name):
    notebook = json.loads(
        (PROJECT_ROOT / "notebooks" / notebook_name).read_text(encoding="utf-8")
    )
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            for node in ast.walk(ast.parse("".join(cell["source"]))):
                if isinstance(node, ast.Assign):
                    if any(isinstance(item, ast.Name) and item.id == variable_name
                           for item in node.targets):
                        return ast.literal_eval(node.value)
    raise ValueError(f"{variable_name} was not found in {notebook_name}.")


def read_validation_table(notebook_name, required_columns):
    notebook = json.loads(
        (PROJECT_ROOT / "notebooks" / notebook_name).read_text(encoding="utf-8")
    )
    for cell in notebook["cells"]:
        for output in cell.get("outputs", []):
            html = "".join(output.get("data", {}).get("text/html", []))
            for table in BeautifulSoup(html, "html.parser").find_all("table"):
                columns = [item.get_text(strip=True) for item in table.select("thead th")]
                columns = [name for name in columns if name]
                if set(required_columns).issubset(columns):
                    rows = [
                        [item.get_text(strip=True) for item in row.find_all("td")]
                        for row in table.select("tbody tr")
                    ]
                    return pd.DataFrame(rows, columns=columns)
    raise ValueError(f"Save {notebook_name} with its validation table outputs first.")


configurations = {
    "XGBoost": read_notebook_setting("11_machine_learning_models.ipynb", "XGBOOST_CONFIGS"),
    "LSTM": read_notebook_setting("12_deep_model.ipynb", "LSTM_CONFIGS"),
}
validation_tables = {
    "XGBoost": read_validation_table(
        "11_machine_learning_models.ipynb",
        ["Country", "Configuration", "Validation Month", "N Series", "MAE"],
    ),
    "LSTM": read_validation_table(
        "12_deep_model.ipynb",
        ["Country", "Configuration", "Validation Month", "Validation MAE",
         "Best Epoch", "N Validation Sequences"],
    ),
}

configuration_rows = []
temporal_rows = []
for model in ["XGBoost", "LSTM"]:
    table = validation_tables[model].copy()
    score = "MAE" if model == "XGBoost" else "Validation MAE"
    count = "N Series" if model == "XGBoost" else "N Validation Sequences"
    table[score] = pd.to_numeric(table[score], errors="raise")
    table[count] = pd.to_numeric(table[count], errors="raise")
    assert table["Validation Month"].eq("2026-02").all()
    assert not table.duplicated(["Country", "Configuration"]).any()
    assert set(table.Country) == set(COUNTRIES)
    assert np.isfinite(table[score]).all() and table[score].ge(0).all()

    for country in COUNTRIES:
        march_row = benchmark.loc[
            benchmark.Country.eq(country) & benchmark.Model.eq(model)
        ].iloc[0]
        selected_name = march_row["Model label"].split("(")[-1].rstrip(")")
        candidates = table.loc[table.Country.eq(country)]
        assert set(candidates.Configuration) == set(configurations[model])
        assert candidates[count].eq(march_row["N Series"]).all()
        best_name = candidates.sort_values(score, kind="stable").iloc[0].Configuration
        assert selected_name == best_name, "March and February configuration records disagree."

        for _, row in candidates.iterrows():
            params = configurations[model][row.Configuration]
            epochs = int(row["Best Epoch"]) if model == "LSTM" else None
            if model == "LSTM":
                assert 1 <= epochs <= read_notebook_setting("12_deep_model.ipynb", "EPOCHS")
            record = {
                "Country": country, "Model": model, "Configuration": row.Configuration,
                "Validation Month": "2026-02", "Validation MAE": float(row[score]),
                "N Validation Series": int(row[count]), "Best Epoch": epochs,
                "Selected": row.Configuration == selected_name,
                **params,
            }
            configuration_rows.append(record)
            if record["Selected"]:
                temporal_rows.append({
                    "Country": country, "Model": model, "Configuration": selected_name,
                    "Validation Month": "2026-02", "Test Month": "2026-03",
                    "Validation MAE": float(row[score]), "March MAE": float(march_row.MAE),
                    "N Validation Series": int(row[count]),
                    "N Test Series": int(march_row["N Series"]),
                    "Best Epoch": epochs,
                })

configuration_results = pd.DataFrame(configuration_rows)
temporal_comparison = pd.DataFrame(temporal_rows)
assert len(configuration_results) == len(COUNTRIES) * 6
assert len(temporal_comparison) == len(COUNTRIES) * 2
temporal_comparison["MAE difference"] = (
    temporal_comparison["March MAE"] - temporal_comparison["Validation MAE"]
)
temporal_comparison["MAE change (%)"] = (
    100 * temporal_comparison["MAE difference"]
    / temporal_comparison["Validation MAE"].replace(0, np.nan)
)
display(configuration_results)
display(temporal_comparison)

## 5. RQ4 — Configuration Complexity and Validation Performance

Within each family, compare the three predefined configurations and highlight the lowest February MAE. XGBoost labels show tree depth, tree count and learning rate; LSTM labels show hidden units, layer count, learning rate and the validation-selected epoch.

Configuration order reflects increasing architecture capacity, not a universal numerical complexity scale. Learning rates and LSTM training durations also differ, so these comparisons do not isolate the causal effect of model size. Parameter counts are not compared across local statistical models and pooled neural/tree models.

In [ ]:
fig, axes = plt.subplots(2, len(COUNTRIES), figsize=(16, 9))
for row_index, model in enumerate(["XGBoost", "LSTM"]):
    order = list(configurations[model])
    for column_index, country in enumerate(COUNTRIES):
        ax = axes[row_index, column_index]
        group = configuration_results.loc[
            configuration_results.Country.eq(country)
            & configuration_results.Model.eq(model)
        ].set_index("Configuration").reindex(order)
        x = np.arange(len(order))
        ax.plot(x, group["Validation MAE"], marker="o", color=MODEL_COLORS[model])
        selected_position = int(np.flatnonzero(group.Selected.to_numpy())[0])
        ax.scatter(
            [selected_position], [group["Validation MAE"].iloc[selected_position]],
            marker="*", s=180, color="#2ca02c", edgecolor="black",
            linewidth=0.5, zorder=3, label="Selected by February MAE",
        )
        labels = []
        for name in order:
            params = configurations[model][name]
            if model == "XGBoost":
                labels.append(
                    f"{name}\n{params['n_estimators']} trees, depth {params['max_depth']}"
                    f"\nlr {params['learning_rate']:g}"
                )
            else:
                epoch = int(group.loc[name, "Best Epoch"])
                labels.append(
                    f"{name}\n{params['hidden_size']} units, {params['num_layers']} layer(s)"
                    f"\nlr {params['learning_rate']:g}; epoch {epoch}"
                )
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_title(f"{country} — {model}")
        ax.set_xlabel("Configuration")
        ax.set_ylabel("February validation MAE (source units)")
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        ax.grid(axis="x", visible=False)
        ax.legend(fontsize=8)
fig.suptitle("RQ4: configuration capacity and February performance", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "rq4_configuration_performance.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

## 6. RQ4 — Validation-to-Test Performance

Connect February validation MAE and March test MAE for each country's selected XGBoost and LSTM configuration. February models were fitted through January; March models were refitted through February. Values are compared within countries, with no pooling of raw errors across country scales.

This is a descriptive temporal comparison, not a conventional training-versus-test generalization gap. Different target-month conditions and the additional training month may affect scores. An increase alone does not establish overfitting. March performance is unavailable for unselected candidates, so this figure cannot establish that the selected architecture was optimal on March.

In [ ]:
fig, axes = plt.subplots(1, len(COUNTRIES), figsize=(15, 5))
for ax, country in zip(axes, COUNTRIES):
    for model in ["XGBoost", "LSTM"]:
        row = temporal_comparison.loc[
            temporal_comparison.Country.eq(country)
            & temporal_comparison.Model.eq(model)
        ].iloc[0]
        values = [row["Validation MAE"], row["March MAE"]]
        ax.plot([0, 1], values, marker="o", label=f"{model}: {row.Configuration}",
                color=MODEL_COLORS[model])
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["February validation\nfit through January",
                        "March test\nrefit through February"])
    ax.set_xlim(-0.15, 1.15)
    ax.set_title(country)
    ax.set_ylabel("MAE (source trade-value units)")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(axis="x", visible=False)
    ax.legend(fontsize=8)
fig.suptitle("RQ4: selected-model performance across the two evaluation months", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "rq4_validation_test_mae.png", dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

configuration_results.to_csv(RESULTS_DIR / "rq4_configuration_results.csv", index=False)
temporal_comparison.to_csv(RESULTS_DIR / "rq4_validation_test_comparison.csv", index=False)
display(temporal_comparison[
    ["Country", "Model", "Configuration", "Validation MAE", "March MAE",
     "MAE difference", "MAE change (%)"]
])

## 7. Descriptive Findings and Interpretation

RQ1 is answered by the March accuracy and metric-specific rankings. RQ4 combines the within-family configuration evidence with validation-to-test stability. A larger architecture is not assumed to be better, and poor March results do not identify their cause.

The printed summaries describe the observed periods only. No significance tests, uncertainty intervals, runtime or memory comparisons are inferred from these aggregate records. Configuration evaluation covers a small predefined set; generalization across months, countries beyond the sample or alternative model settings is not established.

In [ ]:
print("RQ1 — March comparison with classical methods")
for country in COUNTRIES:
    group = benchmark.loc[benchmark.Country.eq(country)]
    classical = group.loc[group.Model.isin(["ETS", "ARIMA"])]
    best_classical = classical.sort_values("sMAPE (%)").iloc[0]
    for model in ["XGBoost", "LSTM"]:
        row = group.loc[group.Model.eq(model)].iloc[0]
        delta = row["sMAPE (%)"] - best_classical["sMAPE (%)"]
        outcome = "lower" if delta < 0 else "higher" if delta > 0 else "equal"
        print(
            f"{country}, {model}: sMAPE {row['sMAPE (%)']:.2f}%; "
            f"{outcome} than {best_classical.Model} "
            f"({best_classical['sMAPE (%)']:.2f}%), by {abs(delta):.2f} percentage points."
        )

print("\nRQ4 — observed configuration choices and temporal changes")
for _, row in temporal_comparison.sort_values(["Country", "Model"]).iterrows():
    change = row["MAE change (%)"]
    change_text = f"{change:+.2f}%" if pd.notna(change) else "undefined (zero validation MAE)"
    print(
        f"{row.Country}, {row.Model}: selected {row.Configuration}; "
        f"February MAE {row['Validation MAE']:,.2f}, "
        f"March MAE {row['March MAE']:,.2f}; change {change_text}."
    )

print("\nSaved outputs directly under:", RESULTS_DIR)
for filename in [
    "rq1_benchmark_results.csv", "rq1_smape_comparison.png", "rq1_metric_rankings.png",
    "rq4_configuration_results.csv", "rq4_configuration_performance.png",
    "rq4_validation_test_comparison.csv", "rq4_validation_test_mae.png",
]:
    assert (RESULTS_DIR / filename).is_file()
    print(filename)

## Summary

The benchmark figures compare six methods on identical March cohort sizes within each country, using sMAPE and rankings across all three metrics. The complexity figures examine three XGBoost and three LSTM configurations using their saved February evidence and then track the selected configurations into March.

Results remain conditional on the existing retrospective cohort selection, one validation month, one test month and rounded saved validation records. The analysis supports descriptive findings about accuracy and whether the tested complexity choices paid off in these periods, not broad claims of generalization or computational efficiency. Original result tables are preserved.

## 8. RQ1 — Interpretation of Forecasting Model Performance

**How do machine learning models compare to classical time series approaches in forecasting trade flows?**

The March 2026 results favour classical methods and simple baselines over the tested XGBoost and LSTM implementations, especially under the primary metric, sMAPE. This conclusion concerns these implementations and this forecast month; it does not establish that machine learning is generally unsuitable for trade forecasting.

**Numerical evidence and country differences**

The comparison covers 27,823 South Korean, 21,973 Slovak and 35,843 US series. Within each country, all six methods report the same number of March observations. The following table gives the exact ordering behind `rq1_smape_comparison.png`, rounded to two decimals.

| Model | South Korea sMAPE (%) | Slovakia sMAPE (%) | USA sMAPE (%) |
|---|---:|---:|---:|
| Naive | **74.62** | 79.87 | 75.88 |
| Seasonal Naive | 78.92 | 85.38 | 77.52 |
| ETS | 97.74 | 74.97 | 74.74 |
| ARIMA | 95.02 | **70.26** | **72.32** |
| XGBoost | 101.69 | 88.86 | 79.03 |
| LSTM | 108.69 | 79.85 | 84.35 |

For **South Korea**, Naive is best on sMAPE (74.62%) and RMSE (22,999,269.32), whereas ETS has the lowest MAE (1,338,033.76). XGBoost and LSTM have sMAPE values 27.07 and 34.07 percentage points above Naive. Both also have higher MAE and RMSE than ETS and ARIMA. Thus the weaker machine learning performance is supported by all three metrics, although the best baseline or classical method depends on the criterion. Naive outperforming Seasonal Naive suggests that the latest observed level was a more useful benchmark than the previous year's corresponding month for this test; it does not demonstrate an absence of annual seasonality.

For **Slovakia**, ARIMA has the lowest sMAPE (70.26%), followed by ETS (74.97%). LSTM achieves 79.85%, almost identical to Naive's 79.87%: its advantage is only 0.021 percentage points and should not be treated as a substantive improvement. XGBoost is last on every metric, with sMAPE of 88.86%. LSTM's MAE of 240,096.69 is slightly below ARIMA's 241,810.70, but its RMSE is considerably higher (3,217,984.40 versus 2,046,319.01). Naive has the lowest MAE (224,437.83) and RMSE (1,829,874.73). LSTM therefore remains competitive on average absolute error, but its squared-error result indicates a greater penalty from large misses.

For the **USA**, ARIMA has the lowest sMAPE (72.32%), followed by ETS (74.74%), while XGBoost and LSTM reach 79.03% and 84.35%. However, ETS is best on MAE (2,743,560.64) and RMSE (27,167,948.15). ARIMA's MAE is 5,573,712.97 and its RMSE is 539,757,252.88, the highest RMSE of all six US methods. ARIMA therefore cannot be called the unqualified best US model: its favourable average proportional error coexists with severe errors in trade-value units. The aggregate results are consistent with very large misses on part of the sample, but do not identify the affected series or prove whether numerical instability, unusual trade movements or another mechanism produced them.

Relative to the **best classical sMAPE** in each country, XGBoost is worse by 6.67 percentage points in South Korea, 18.60 in Slovakia and 6.71 in the USA. LSTM is worse by 13.67, 9.60 and 12.04 percentage points, respectively. LSTM nevertheless has lower MAE and RMSE than XGBoost in all three countries. Their sMAPE ordering differs: XGBoost is better in South Korea and the USA, while LSTM is better in Slovakia.

**Why the metrics give different answers**

The ranking heatmap, `rq1_metric_rankings.png`, shows why reporting one score alone would be misleading. MAE measures average absolute error in source trade-value units; large-value series can contribute much larger errors. RMSE further emphasises large deviations by squaring errors. sMAPE instead scales each error by the magnitude of the actual and forecast, is bounded between 0% and 200%, and then averages across observations. It is not a percentage of total national trade value. A zero actual with a nonzero forecast receives 200%, while an actual and forecast both equal to zero receive 0% under the project's implementation. Zero and small-value observations can therefore materially affect sMAPE rankings. The differences between metrics reflect different forecasting priorities rather than a contradiction.

**Plausible interpretation, assumptions and limitations**

One plausible explanation is that recent series-specific levels are difficult to beat at a one-month horizon, while pooled machine learning models must learn across heterogeneous partners, products and trade flows. The implementations also differ in predictors, fitting objectives and transformations: XGBoost uses squared error on trade values, while LSTM uses log-transformed, scaled values and Huber loss. These mechanisms may help explain the observed differences, but the saved aggregate scores do not isolate their effects. The results compare complete modelling pipelines, rather than architecture alone.

Missing monthly trade observations are assumed to be zero, as specified for this project; missing reports would be treated as no trade under that assumption. The retained cohorts also reflect the original retrospective selection, including the exclusion of series with more than 90% zero months. Selection was not reconstructed using only information available before the forecast origin. Consequently, these are conditional results for the retained historical sample, rather than a fully prospective evaluation of which series would have been available for selection at that time.

Identical counts and the notebook's checks against March target keys support comparability, but the original saved aggregate results do not contain keyed predictions with which to independently audit every evaluated pair or inspect extreme residuals. Raw MAE and RMSE should be compared within countries, not pooled across different trade-value scales. Only one test month is observed, with no uncertainty intervals, repeated monthly evaluation or significance tests. Candidate selection for machine learning used February MAE, whereas the principal reported ranking uses March sMAPE, so selection and reporting objectives differ.

The fixed March target and available computation motivated a single chronological February holdout, three machine learning candidates per family and reuse of the original fitted benchmark results. These are practical scope choices rather than inherently incorrect forecasting logic: a chronological holdout is valid for the specified one-month task. They do, however, limit evidence about tuning quality and stability across time. Comparing pipelines with different objectives is also legitimate for benchmarking, provided it is not presented as an isolated architecture effect. In contrast, retrospective cohort selection is a genuine prospective-selection weakness, and assumed publication availability is unverified; resource limits and preservation of the original workflow explain why a full redesign was not undertaken, but do not make those issues disappear.

**Main findings and answer**

- Neither machine learning method improves on the best classical method's March sMAPE in any country.
- Simple forecasting remains competitive: Naive wins South Korean sMAPE and Slovak MAE/RMSE; ETS wins US MAE/RMSE.
- LSTM performs better than XGBoost on MAE and RMSE throughout, but neither has a consistent advantage across all metrics.
- The strongest supported conclusion is that additional modelling flexibility did not deliver a consistent accuracy advantage in this experiment. Model choice should reflect the country and error criterion; US ARIMA illustrates why a strong sMAPE result must be checked against errors in trade-value units.


## 9. RQ4 — Interpretation of Complexity and Generalization

**What is the trade-off between model complexity and generalization performance?**

Greater model capacity does not produce a consistent improvement in the available evidence. Some larger configurations improve February validation accuracy, while others are worse; the selected machine learning configurations also generally have higher MAE in March. These observations support selective, country-specific use of complexity, but do not establish a causal relationship between model size and generalization.

**Configuration-level evidence**

The figure `rq4_configuration_performance.png` compares three configurations per family. XGBoost_1 uses 200 trees of maximum depth 4 with learning rate 0.05; XGBoost_2 uses 300 trees, depth 6 and rate 0.05; XGBoost_3 uses 300 trees, depth 8 and rate 0.03. LSTM_1 uses 16 hidden units in one layer at rate 0.001; LSTM_2 uses 32 units in one layer at rate 0.001; LSTM_3 uses 32 units in two layers at rate 0.0005. Lower February MAE determines selection.

| Country | XGBoost_1 MAE | XGBoost_2 MAE | XGBoost_3 MAE | Selected |
|---|---:|---:|---:|---|
| South Korea | 1,724,899.00 | 1,651,236.00 | **1,645,711.00** | XGBoost_3 |
| Slovakia | **291,797.20** | 299,742.40 | 295,396.50 | XGBoost_1 |
| USA | 3,923,460.00 | 3,924,299.00 | **3,911,390.00** | XGBoost_3 |

The largest XGBoost configuration improves on the smallest by 4.59% in South Korea, but only 0.31% in the USA. In Slovakia, the smallest configuration wins, and increasing capacity does not help. The US plot's narrow vertical range makes its differences visually prominent even though their relative magnitudes are small; the percentage change is needed to assess practical importance.

| Country | LSTM_1 MAE | LSTM_2 MAE | LSTM_3 MAE | Selected |
|---|---:|---:|---:|---|
| South Korea | 1,303,469.00 | **1,058,063.00** | 1,180,645.00 | LSTM_2 |
| Slovakia | 283,658.10 | 448,693.40 | **250,027.60** | LSTM_3 |
| USA | **2,702,092.00** | 2,838,419.00 | 2,710,258.00 | LSTM_1 |

For LSTM, moving from the smallest to the selected architecture reduces validation MAE by 18.83% in South Korea and 11.86% in Slovakia. South Korea favours a wider single layer rather than two layers; Slovakia favours the two-layer configuration; the USA favours the smallest model. The Slovak two-layer configuration improves substantially over the wider single-layer configuration, but also uses a different learning rate. There is no universal progression in which larger models are consistently better.

Training duration is another part of the comparison. The selected LSTM epochs are 6 for South Korea, 1 for Slovakia and 9 for the USA. All Slovak candidates select epoch 1. This indicates that later epochs did not improve the saved February selection criterion within the training run; it does not establish that a single epoch would be optimal for other periods or seeds. Architecture, learning rate and selected duration jointly determine these outcomes.

**Performance after refitting for March**

The figure `rq4_validation_test_mae.png` tracks the selected configuration in each country. February forecasts use fitting data through January; March forecasts use a fresh fit through February.

| Country | Selected configuration | February MAE | March MAE | Change in MAE (%) |
|---|---|---:|---:|---:|
| South Korea | XGBoost_3 | 1,645,711.00 | 2,013,894.84 | +22.37 |
| Slovakia | XGBoost_1 | 291,797.20 | 354,806.97 | +21.59 |
| USA | XGBoost_3 | 3,911,390.00 | 4,326,257.43 | +10.61 |
| South Korea | LSTM_2 | 1,058,063.00 | 1,616,568.80 | +52.79 |
| Slovakia | LSTM_3 | 250,027.60 | 240,096.69 | -3.97 |
| USA | LSTM_1 | 2,702,092.00 | 3,764,889.20 | +39.33 |

XGBoost MAE increases in all three countries. LSTM increases more sharply in South Korea and the USA, but improves in Slovakia. Thus five of six selected pipelines have higher MAE in March. LSTM still has lower MAE than XGBoost in every country in both months, so a larger temporal deterioration does not automatically imply a worse final score. Nor is the largest architecture necessarily the least stable: the two-layer Slovak LSTM is the only selected pipeline whose MAE decreases.

**What this does and does not establish**

February and March are both held-out forecast targets at their respective origins. Their difference is a temporal performance change, not a training-versus-test generalization gap. Different trade values, a different month length, unusual movements, the extra training month and stochastic refitting can affect it. Higher March MAE alone is insufficient evidence of overfitting or concept drift. Since configuration selection minimises February error, its reported validation score may also be optimistic relative to a fresh evaluation period.

The comparison varies several settings simultaneously, including learning rates, tree count and LSTM training duration. Consequently, it cannot isolate the effect of depth or parameter count. Only three predefined candidates are tested per family, one validation month is used, and saved validation figures have limited displayed precision. There are no repeated-seed estimates or uncertainty intervals. March scores for the unselected configurations are unavailable, so the study cannot establish whether a smaller or larger alternative would have generalised better in March.

The RQ1 benchmark adds useful context: the tested flexible models do not displace the strongest simpler alternatives consistently. However, local ETS/ARIMA fits and pooled XGBoost/LSTM fits use different representations and objectives; there is no common numerical complexity scale across them. 

Limited compute and the decision to preserve completed March benchmarks motivated a small candidate grid, one chronological validation month and evaluation of only the selected configurations on March. This is a valid economical selection procedure, not a logical error, but it leaves the March performance of unselected candidates unknown. Varying several hyperparameters together is acceptable for choosing practical configurations; it cannot isolate the effect of model size. Different February and March scores are legitimate descriptive evidence, whereas treating their difference as proof of overfitting would be an unsupported inference. Runtime and memory were not systematically measured, so no numerical computational-efficiency trade-off can be claimed. The inherited cohort-selection and data-availability caveats from RQ1 also remain.

**Main findings and answer**

- Useful capacity depends on the country and model family: the selected LSTM architectures range from the smallest to the largest, and Slovakia selects the smallest XGBoost.
- Additional capacity can improve validation accuracy, but improvements range from substantial to very small and are not monotonic.
- Most selected models deteriorate from February to March, yet the evidence cannot attribute this to overfitting or show that complexity caused the change.
- The practical implication is to retain simple benchmarks and justify added capacity through held-out gains. The available experiment supports country-specific configuration choices rather than a general claim that greater complexity yields stronger generalization; stability across more forecast origins remains untested.
